In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
# ============================================================
# Smart MCQ Solver – Full Multi-Model Pipeline
# ============================================================
# Requirements:
#   • Model built from scratch     → TF-IDF + Logistic Regression
#   • Pretrained model             → Qwen2.5-7B (4-bit) embeddings + LR
#   • Additional models of choice  → CatBoost, XGBoost, LightGBM, RandomForest
#   • Ensemble of the tree models  + overall weighted ensemble
# ============================================================

# ---------- Installs ----------
!pip uninstall -y scikit-learn -q
!pip install -q scikit-learn==1.5.2
!pip install -q sentence-transformers catboost xgboost lightgbm wandb bitsandbytes accelerate --upgrade

import os, gc, warnings, traceback
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import wandb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# ============================================================
# Config
# ============================================================
CONFIG = {
    "train_path": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    "test_path":  "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv",
    "out_path":   "/kaggle/working/submission.csv",
    "max_features": 30000,
    "n_splits": 5,
    "wandb_project": "23f2004250-t22026",
    "wandb_run_name": "qwen-tree-ensemble",
    "wandb_key": "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",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "qwen_model": "Qwen/Qwen2.5-7B",          # will be loaded in 4-bit
    "fallback_st_model": "sentence-transformers/all-MiniLM-L6-v2",
}

LETTERS = ["A", "B", "C", "D", "E"]

# ============================================================
# Helpers
# ============================================================
def build_text(df):
    return (
        df["prompt"].fillna("") + " " + df["prompt"].fillna("") + " "
        + "[A] " + df["A"].fillna("") + " "
        + "[B] " + df["B"].fillna("") + " "
        + "[C] " + df["C"].fillna("") + " "
        + "[D] " + df["D"].fillna("") + " "
        + "[E] " + df["E"].fillna("")
    )

def map_at_3(y_true, proba):
    top3 = np.argsort(-proba, axis=1)[:, :3]
    scores = np.zeros(len(y_true))
    for k in range(3):
        scores += (top3[:, k] == y_true) / (k + 1)
    return scores.mean()

def proba_to_top3(proba):
    idx = np.argsort(-proba, axis=1)[:, :3]
    return [" ".join(LETTERS[i] for i in row) for row in idx]

def gpu_available():
    try:
        import subprocess
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception:
        return False

# ============================================================
# 1. Load data
# ============================================================
wandb.login(key=CONFIG["wandb_key"])
wandb.init(project=CONFIG["wandb_project"], name=CONFIG["wandb_run_name"], config=CONFIG)

print("Loading data...")
train = pd.read_csv(CONFIG["train_path"])
test  = pd.read_csv(CONFIG["test_path"])
train["answer"] = train["answer"].astype(str).str.strip().str.upper()

le = LabelEncoder()
le.fit(LETTERS)
y = le.transform(train["answer"])

print(f"Train: {len(train)} | Test: {len(test)}")

# ============================================================
# 2. TF-IDF features (shared by LR-from-scratch + all tree models)
# ============================================================
print("\nBuilding TF-IDF features...")
train_text = build_text(train)
test_text  = build_text(test)

tfidf = TfidfVectorizer(
    max_features=CONFIG["max_features"],
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    stop_words="english"
)
X_tfidf_train = tfidf.fit_transform(train_text)
X_tfidf_test  = tfidf.transform(test_text)
print("TF-IDF shape:", X_tfidf_train.shape)

# ============================================================
# 3. Pretrained model – Qwen2.5-7B (4-bit) or MiniLM fallback
# ============================================================
def get_qwen_embeddings(texts, model_name, batch_size=4, max_len=256):
    """Extract mean-pooled embeddings from Qwen2.5-7B in 4-bit."""
    print(f"Loading {model_name} in 4-bit...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_len, return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)
            # mean pool
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            summed = (outputs.last_hidden_state * mask).sum(1)
            counts = mask.sum(1).clamp(min=1e-9)
            emb = summed / counts
            emb = F.normalize(emb, p=2, dim=1)
            all_embs.append(emb.cpu().numpy())

        if i % 64 == 0:
            print(f"  processed {i}/{len(texts)}")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return np.concatenate(all_embs, axis=0)

print("\nExtracting pretrained embeddings...")
use_qwen = False
try:
    emb_train = get_qwen_embeddings(train_text.tolist(), CONFIG["qwen_model"])
    emb_test  = get_qwen_embeddings(test_text.tolist(),  CONFIG["qwen_model"])
    use_qwen = True
    print("Successfully used Qwen2.5-7B embeddings")
except Exception as e:
    print("Qwen2.5-7B failed (likely OOM or missing bitsandbytes). Falling back to MiniLM.")
    print("Error:", str(e)[:200])
    st = SentenceTransformer(CONFIG["fallback_st_model"], device=CONFIG["device"])
    emb_train = st.encode(train_text.tolist(), batch_size=128, show_progress_bar=True, convert_to_numpy=True)
    emb_test  = st.encode(test_text.tolist(),  batch_size=128, show_progress_bar=True, convert_to_numpy=True)
    del st
    gc.collect()
    torch.cuda.empty_cache()
    print("Using MiniLM embeddings as fallback")

print("Embedding shape:", emb_train.shape)

# ============================================================
# 4. Define all models
# ============================================================
# --- Model built from scratch ---
lr_tfidf = LogisticRegression(max_iter=1000, C=2.0, solver="saga", n_jobs=-1, random_state=42)

# --- Pretrained model head ---
lr_pretrained = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", n_jobs=-1, random_state=42)

# --- Additional models of choice ---
catboost = CatBoostClassifier(
    iterations=800, depth=6, learning_rate=0.05,
    loss_function="MultiClass", random_seed=42, verbose=False,
    task_type="GPU" if gpu_available() else "CPU"
)
xgboost = XGBClassifier(
    n_estimators=800, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective="multi:softprob", num_class=5,
    eval_metric="mlogloss", tree_method="hist",
    random_state=42, n_jobs=-1, verbosity=0
)
lightgbm = LGBMClassifier(
    n_estimators=800, num_leaves=63, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective="multiclass", num_class=5,
    random_state=42, n_jobs=-1, verbose=-1
)
rf = RandomForestClassifier(
    n_estimators=600, max_depth=None, min_samples_leaf=2,
    n_jobs=-1, random_state=42
)

# Group them
from_scratch = {"lr_tfidf": (lr_tfidf, X_tfidf_train, X_tfidf_test)}
pretrained   = {"lr_pretrained": (lr_pretrained, emb_train, emb_test)}
additional   = {
    "catboost": (catboost, X_tfidf_train, X_tfidf_test),
    "xgboost":  (xgboost,  X_tfidf_train, X_tfidf_test),
    "lightgbm": (lightgbm, X_tfidf_train, X_tfidf_test),
    "rf":       (rf,       X_tfidf_train, X_tfidf_test),
}

all_models = {**from_scratch, **pretrained, **additional}

# ============================================================
# 5. Cross-validation
# ============================================================
print("\nRunning 5-fold CV on all models...")
skf = StratifiedKFold(n_splits=CONFIG["n_splits"], shuffle=True, random_state=42)

oof = {name: np.zeros((len(y), 5)) for name in all_models}
fold_scores = {name: [] for name in all_models}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tfidf_train, y)):
    print(f"\n--- Fold {fold+1} ---")
    for name, (template, X_all, _) in all_models.items():
        X_tr, X_va = X_all[tr_idx], X_all[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        # fresh instance
        if name.startswith("lr"):
            m = LogisticRegression(**template.get_params())
        elif name == "catboost":
            m = CatBoostClassifier(**template.get_params())
        elif name == "xgboost":
            m = XGBClassifier(**template.get_params())
        elif name == "lightgbm":
            m = LGBMClassifier(**template.get_params())
        else:
            m = clone(template)

        if name == "catboost":
            m.fit(X_tr, y_tr, verbose=False)
        else:
            m.fit(X_tr, y_tr)

        proba = m.predict_proba(X_va)
        oof[name][va_idx] = proba
        m3 = map_at_3(y_va, proba)
        fold_scores[name].append(m3)

        wandb.log({f"{name}/fold": fold, f"{name}/map_at_3": m3})
        print(f"  {name:14s} MAP@3 = {m3:.5f}")

# Summary
print("\n=== OOF MAP@3 Summary ===")
summary_rows = []
for name in all_models:
    mean_m3 = np.mean(fold_scores[name])
    std_m3  = np.std(fold_scores[name])
    summary_rows.append({"model": name, "mean_map_at_3": mean_m3, "std": std_m3})
    print(f"{name:14s}: {mean_m3:.5f} ± {std_m3:.5f}")
    wandb.log({f"{name}/oof_map_at_3": mean_m3})

summary_df = pd.DataFrame(summary_rows).sort_values("mean_map_at_3", ascending=False)
wandb.log({"model_comparison": wandb.Table(dataframe=summary_df)})
print(summary_df.to_string(index=False))

# ============================================================
# 6. Ensemble the additional (tree) models
# ============================================================
print("\n--- Tree models ensemble ---")
tree_names = list(additional.keys())
tree_scores = {n: np.mean(fold_scores[n]) for n in tree_names}
tree_total = sum(tree_scores.values())
tree_weights = {n: s / tree_total for n, s in tree_scores.items()}

tree_oof = np.zeros((len(y), 5))
for n, w in tree_weights.items():
    tree_oof += w * oof[n]

tree_map3 = map_at_3(y, tree_oof)
print("Tree ensemble weights:", {k: round(v, 3) for k, v in tree_weights.items()})
print(f"Tree ensemble OOF MAP@3: {tree_map3:.5f}")
wandb.log({"tree_ensemble/oof_map_at_3": tree_map3})

# ============================================================
# 7. Overall ensemble (from-scratch + pretrained + tree-ensemble)
# ============================================================
print("\n--- Overall ensemble ---")
# We treat the tree ensemble as one strong model
overall_scores = {
    "lr_tfidf": np.mean(fold_scores["lr_tfidf"]),
    "lr_pretrained": np.mean(fold_scores["lr_pretrained"]),
    "tree_ensemble": tree_map3,
}
overall_total = sum(overall_scores.values())
overall_weights = {k: v / overall_total for k, v in overall_scores.items()}

overall_oof = (
    overall_weights["lr_tfidf"]      * oof["lr_tfidf"] +
    overall_weights["lr_pretrained"] * oof["lr_pretrained"] +
    overall_weights["tree_ensemble"] * tree_oof
)
overall_map3 = map_at_3(y, overall_oof)

print("Overall weights:", {k: round(v, 3) for k, v in overall_weights.items()})
print(f"Overall OOF MAP@3: {overall_map3:.5f}")
wandb.log({"overall_ensemble/oof_map_at_3": overall_map3})

# ============================================================
# 8. Refit everything on full data & create submission
# ============================================================
print("\nRefitting on full data and predicting test set...")

# from-scratch
m_lr = LogisticRegression(**lr_tfidf.get_params())
m_lr.fit(X_tfidf_train, y)
proba_lr = m_lr.predict_proba(X_tfidf_test)

# pretrained head
m_pre = LogisticRegression(**lr_pretrained.get_params())
m_pre.fit(emb_train, y)
proba_pre = m_pre.predict_proba(emb_test)

# tree models
tree_test_proba = np.zeros((len(test), 5))
for name, (template, X_tr, X_te) in additional.items():
    if name == "catboost":
        m = CatBoostClassifier(**template.get_params())
        m.fit(X_tr, y, verbose=False)
    elif name == "xgboost":
        m = XGBClassifier(**template.get_params())
        m.fit(X_tr, y)
    elif name == "lightgbm":
        m = LGBMClassifier(**template.get_params())
        m.fit(X_tr, y)
    else:
        m = clone(template)
        m.fit(X_tr, y)
    tree_test_proba += tree_weights[name] * m.predict_proba(X_te)

# final blend
final_proba = (
    overall_weights["lr_tfidf"]      * proba_lr +
    overall_weights["lr_pretrained"] * proba_pre +
    overall_weights["tree_ensemble"] * tree_test_proba
)

preds = proba_to_top3(final_proba)
submission = pd.DataFrame({"ID": test["id"], "Prediction": preds})
submission.to_csv(CONFIG["out_path"], index=False)

print(f"\nSubmission saved → {CONFIG['out_path']}")
print(submission.head(10))
print(f"\nFinal OOF MAP@3 (overall ensemble): {overall_map3:.5f}")

wandb.log({"final/oof_map_at_3": overall_map3})
wandb.finish()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 82.5 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 11.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 18.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 77.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 57.3 MB/s eta 0

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2004250 (23f2004250-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading data...
Train: 2000 | Test: 500

Building TF-IDF features...


TF-IDF shape: (2000, 11528)

Extracting pretrained embeddings...
Loading Qwen/Qwen2.5-7B in 4-bit...


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

...

  processed 0/2000
  processed 64/2000
  processed 128/2000
  processed 192/2000
  processed 256/2000
  processed 320/2000
  processed 384/2000
  processed 448/2000
  processed 512/2000
  processed 576/2000
  processed 640/2000
  processed 704/2000
  processed 768/2000
  processed 832/2000
  processed 896/2000
  processed 960/2000
  processed 1024/2000
  processed 1088/2000
  processed 1152/2000
  processed 1216/2000
  processed 1280/2000
  processed 1344/2000
  processed 1408/2000
  processed 1472/2000
  processed 1536/2000
  processed 1600/2000
  processed 1664/2000
  processed 1728/2000
  processed 1792/2000
  processed 1856/2000
  processed 1920/2000
  processed 1984/2000
Loading Qwen/Qwen2.5-7B in 4-bit...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2Model LOAD REPORT from: Qwen/Qwen2.5-7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  processed 0/500
  processed 64/500
  processed 128/500
  processed 192/500
  processed 256/500
  processed 320/500
  processed 384/500
  processed 448/500
Successfully used Qwen2.5-7B embeddings
Embedding shape: (2000, 3584)

Running 5-fold CV on all models...

--- Fold 1 ---
  lr_tfidf       MAP@3 = 1.00000
  lr_pretrained  MAP@3 = 0.63958
  catboost       MAP@3 = 1.00000
  xgboost        MAP@3 = 0.99500
  lightgbm       MAP@3 = 0.99875
  rf             MAP@3 = 0.99708

--- Fold 2 ---
  lr_tfidf       MAP@3 = 1.00000
  lr_pretrained  MAP@3 = 0.66917
  catboost       MAP@3 = 1.00000
  xgboost        MAP@3 = 1.00000
  lightgbm       MAP@3 = 1.00000
  rf             MAP@3 = 1.00000

--- Fold 3 ---
  lr_tfidf       MAP@3 = 1.00000
  lr_pretrained  MAP@3 = 0.68583
  catboost       MAP@3 = 1.00000
  xgboost        MAP@3 = 0.99500
  lightgbm       MAP@3 = 0.99875
  rf             MAP@3 = 0.99500

--- Fold 4 ---
  lr_tfidf       MAP@3 = 1.00000
  lr_pretrained  MAP@3 = 0.65500
  catboost   

catboost/fold,▁▃▅▆█
catboost/map_at_3,▁▁▁▁▁
catboost/oof_map_at_3,▁
final/oof_map_at_3,▁
lightgbm/fold,▁▃▅▆█
lightgbm/map_at_3,▁█▁██
lightgbm/oof_map_at_3,▁
lr_pretrained/fold,▁▃▅▆█
lr_pretrained/map_at_3,▁▅█▃▇
lr_pretrained/oof_map_at_3,▁
+11,...
